## Exploración inicial de ENCOVI 2023

Este notebook realiza un análisis exploratorio de los microdatos de ENCOVI 2023. Se cargan las bases de Personas y Hogares, se verifican las columnas principales y se revisa la estructura de los ponderadores (FACTOR / Factor2).

**Objetivo:** Generar un vistazo rápido a las dimensiones de los datos, valores faltantes y estadísticas descriptivas por departamento, sentando las bases para el cálculo de indicadores de informalidad y pobreza.

In [1]:
import pandas as pd
import pyreadstat
import os
import warnings

warnings.filterwarnings('ignore')

# Búsqueda automática de la carpeta de datos
current = os.getcwd()
data_path = None
for _ in range(5):
    candidate = os.path.join(current, 'data', 'raw', 'encovi_2023')
    if os.path.exists(candidate):
        data_path = candidate
        break
    parent = os.path.dirname(current)
    if parent == current:
        break
    current = parent

if data_path is None:
    print('No se encontró la carpeta data/raw/encovi_2023')
else:
    print('Carpeta de datos encontrada:', data_path)
    df_personas, meta = pyreadstat.read_sav(os.path.join(data_path, 'encovi_2023_personas.sav'))
    print(f'Base de Personas cargada: {df_personas.shape[0]} filas x {df_personas.shape[1]} columnas')

Carpeta de datos encontrada: /private/tmp/ine-faro-I02-integracion/data/raw/encovi_2023


Base de Personas cargada: 46017 filas x 626 columnas


### Muestra de datos

In [2]:
print('Personas:', len(df_personas), 'registros; no se muestran filas individuales.')

Personas: 46017 registros; no se muestran filas individuales.


### Estadísticas descriptivas

In [3]:
df_num = df_personas.select_dtypes(include=['float64', 'int64'])
print(df_num.describe().transpose())

                   count          mean          std   min     25%      50%  \
DEPTO            46017.0     11.071169     6.501444   1.0     5.0     11.0   
DOMINIO          46017.0      2.434535     0.607867   1.0     2.0      2.0   
AREA             46017.0      1.496425     0.499993   1.0     1.0      1.0   
UPM              46017.0  14334.122520  7045.715305  29.0  8013.0  13318.0   
FACTOR           46017.0    374.580438   299.205451  22.0   179.0    276.0   
...                  ...           ...          ...   ...     ...      ...   
OCUPADOS         18036.0      1.000000     0.000000   1.0     1.0      1.0   
DESOCUPADOS        265.0      1.000000     0.000000   1.0     1.0      1.0   
SUBVISIBLES       2225.0      1.000000     0.000000   1.0     1.0      1.0   
INACTIVOS        20761.0      1.000000     0.000000   1.0     1.0      1.0   
FORMAL_INFORMAL  18036.0      1.265968     0.441860   1.0     1.0      1.0   

                     75%      max  
DEPTO               17.0   

### Verificación de ponderadores

In [4]:
print('Conteo de FACTOR no nulo:', df_personas['FACTOR'].notna().sum())
if 'Factor3' in df_personas.columns:
    print('Conteo de Factor3 no nulo:', df_personas['Factor3'].notna().sum())
    print('Correlación Pearson:', df_personas['FACTOR'].corr(df_personas['Factor3']))
elif 'Factor2' in df_personas.columns:
    print('Factor3 no existe. Usando Factor2...')
    print('Conteo de Factor2 no nulo:', df_personas['Factor2'].notna().sum())
    print('Correlación Pearson:', df_personas['FACTOR'].corr(df_personas['Factor2']))
else:
    print('Ningún factor alternativo encontrado.')

Conteo de FACTOR no nulo: 46017
Factor3 no existe. Usando Factor2...
Conteo de Factor2 no nulo: 46017
Correlación Pearson: 0.9976422736957602


## Controles complementarios de I-02

Se conservan las definiciones, cálculos y resultados incorporados en I-03. Las comprobaciones siguientes son estructurales y no recalculan tasas. La correlación exploratoria anterior no autoriza intercambiar pesos. Los conteos sin filtro de edad no deben confundirse con el universo de ocupados de 15 años o más de la metodología.

In [5]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data/raw/encovi_2023').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'scripts/explorar_encovi.py').is_file()
sys.path.insert(0, str(ROOT / 'scripts'))
from explorar_encovi import cargar_bases, crear_diccionario, auditar_bases, guardar_resultados
RAW = ROOT / 'data/raw/encovi_2023'
bases = cargar_bases(RAW, precargadas={'personas': (df_personas, meta)})
print('Cinco bases cargadas; no se imprimen registros individuales.')

Cinco bases cargadas; no se imprimen registros individuales.


In [6]:
informe = auditar_bases(bases)
print(pd.DataFrame(informe['bases']).T[['filas','columnas','unidad','clave','duplicados_clave','faltantes_clave','hogares_distintos']].to_string())
print('Departamentos por base:', pd.DataFrame(informe['cobertura']).groupby('base').depto_id.nunique().to_dict())

            filas columnas             unidad                   clave duplicados_clave faltantes_clave hogares_distintos
personas    46017      626            persona          [NO_HOGAR, CP]                0               0             10964
hogares     10964      129              hogar              [NO_HOGAR]                0               0             10964
consumo     10930       29  hogar con consumo              [NO_HOGAR]                0               0             10930
negocios_1  10964       11              hogar              [NO_HOGAR]                0               0             10964
negocios_2   2490       26  negocio del hogar  [NO_HOGAR, ID_NEGOCIO]                0               0              2142
Departamentos por base: {'consumo': 22, 'hogares': 22, 'negocios_1': 22, 'negocios_2': 22, 'personas': 22}


In [7]:
diccionario, contraste = crear_diccionario(bases, RAW)
print(pd.DataFrame(contraste).T.to_string())
print('Variables publicadas:', int(diccionario.naturaleza.eq('original_publicada').sum()))
print('Propuestas no implementadas:', int(diccionario.naturaleza.eq('derivada_propuesta_no_implementada').sum()))
seleccion = diccionario[diccionario.variable.isin(['POBREZA','pobreza','FORMAL_INFORMAL','FACTOR','Factor2','Factor3'])]
print(seleccion[['base','variable','faltantes_sistema','codigos_observados_sin_etiqueta','n_codigos_sin_etiqueta']].to_string(index=False))

           variables_sav variables_xlsx solo_sav solo_xlsx
personas             626            626       []        []
hogares              129            129       []        []
consumo               29             29       []        []
negocios_1            11             11       []        []
negocios_2            26             26       []        []
Variables publicadas: 821
Propuestas no implementadas: 7
      base        variable faltantes_sistema codigos_observados_sin_etiqueta n_codigos_sin_etiqueta
  personas          FACTOR                 0                              []                      0
  personas         Factor2                 0                              []                      0
  personas         POBREZA                 0                           [0.0]                     95
  personas FORMAL_INFORMAL             27981                              []                      0
   hogares          FACTOR                 0                              []             

In [8]:
categorias = pd.DataFrame(informe['categorias'])
print(categorias[categorias.variable.isin(['POBREZA','pobreza','FORMAL_INFORMAL','PET','PEA','OCUPADOS'])].to_string(index=False))

      base        variable  codigo                        etiqueta  n_no_ponderado
  personas         POBREZA     0.0                Sin etiqueta SAV              95
  personas         POBREZA     1.0                      Pobre ext.            7222
  personas         POBREZA     2.0                  Pobre no extr.           19394
  personas         POBREZA     3.0                        No pobre           19306
  personas FORMAL_INFORMAL     1.0                        Informal           13239
  personas FORMAL_INFORMAL     2.0                          Formal            4797
  personas             PET     1.0   Población en edad de trabajar           39062
  personas             PEA     1.0 Poblacíon económicamente activa           18301
  personas        OCUPADOS     1.0               Población ocupada           18036
   hogares         POBREZA     0.0                Sin etiqueta SAV              34
   hogares         POBREZA     1.0                   Pobre extremo            1173
   h

In [9]:
print(pd.DataFrame(informe['uniones_a_hogares']).T.to_string())
print('Consumo:', json.dumps(informe['consumo'], ensure_ascii=False))
print('Negocios:', json.dumps(informe['negocios'], ensure_ascii=False))

            cardinalidad filas_antes filas_despues filas_sin_hogar hogares_sin_registro_hijo departamento_discordante upm_discordante
personas    muchos_a_uno       46017         46017               0                         0                        0               0
consumo        uno_a_uno       10930         10930               0                        34                        0               0
negocios_1     uno_a_uno       10964         10964               0                         0                        0               0
negocios_2  muchos_a_uno        2490          2490               0                      8822                        0               0
Consumo: {"guia_hogares_con_consumo": 10930, "observados": 10930, "diferencia_con_guia": 0, "hogares_sin_consumo": 34, "personas_en_hogares_sin_consumo": 95, "clasificacion_pobreza_sin_consumo": {"0": 34}, "Factor3_cero_sin_consumo": 34}
Negocios: {"hogares_declaran_negocio": 2142, "hogares_con_detalle": 2142, "declaran_sin_deta

In [10]:
print(pd.DataFrame(informe['pesos']).to_string(index=False))
print(json.dumps(informe['relaciones_pesos'], ensure_ascii=False, indent=2))

      base variable                                               etiqueta  faltantes  ceros  negativos  no_finitos
  personas   FACTOR                                    Factor de Expansión          0      0          0           0
  personas  Factor2 Factor de expansión de hogares con agregado de consumo          0     95          0           0
   hogares   FACTOR                                    Factor de Expansión          0      0          0           0
   hogares  Factor3                       Factor2 * personas (con consumo)          0     34          0           0
   consumo   FACTOR Factor de expansión de hogares con agregado de consumo          0      0          0           0
   consumo  Factor3                                     Factor2 * personas          0      0          0           0
negocios_1   FACTOR                                    Factor de Expansión          0      0          0           0
negocios_2   FACTOR                                    Factor de Expansi

In [11]:
resultado = guardar_resultados(ROOT, diccionario, informe, contraste)
assert resultado['consumo']['observados'] == 10930
assert resultado['tasas_poblacionales_calculadas'] is False
print('Guardados controles y diccionario complementario en docs/evidencia/I-02; el diccionario de main se conserva.')
print('Referencia metodológica conservada:', resultado['referencia_metodologica'])

Guardados controles y diccionario complementario en docs/evidencia/I-02; el diccionario de main se conserva.
Referencia metodológica conservada: docs/metodologia.md


### Alcance de las cantidades de negocios

2,469 suma P14A02A en todos los hogares; 2,440 corresponde sólo a los 2,142 hogares declarantes; 2,490 son registros de detalle, no hogares. Las 73 discrepancias son hogares declarantes con cantidades distintas entre módulos. Los 34 hogares y 95 personas sin Consumo pertenecen a otro control: no se deduce igualdad de conjuntos por sus tamaños. Se conservan las observaciones originales sin imputarlas ni eliminarlas.